# 02_geometry — Representational geometry: SRM, Procrustes disparity and its validity checks

**Manuscript:** Results section 2; Figure 5; Methods 'Shared Response Model', 'Representational geometry'; Supplementary S4 (dimensionality), S8 (LOO disparity), S9 (alignment-independent checks), S10 (activation), S18 (validity of the geometric comparison).

`rerun_loo_consistent.py` fits the control-only shared response model (BrainIAK SRM, k = 4/4/3/3) with leave-one-control-out references and computes Procrustes disparity in the common space and under the symmetric leave-one-subject-out estimator, with Crawford-Howell single-case tests. The `validation/` scripts implement the alignment-independent distances (crossnobis, PCA, PCA-CCA), the colour-correspondence permutation with a frozen projection, the cyclic-shift analysis, and the activation-level comparison. BrainIAK requires `mpirun -np 1 python ...`; the notebook therefore loads the committed outputs and recomputes only the aggregate statistics (Spearman correlations, Hedges g, BH correction).

**How to read this notebook.** Every code cell loads committed result files from `results/` and compares the values it derives with the numbers printed in the manuscript (`V.check`). A check passes when the produced value equals the printed one at the printed precision, or satisfies the stated relation. Quantities that have no committed artifact are recorded as pointers (`V.flag`) rather than silently omitted. The last cell tallies the checks and writes `_checks_02_geometry.json`, which `run_notebooks.py` collects into `REPORT.md`.

Provenance: built by `tools/public_repo/build.py` of the development repository (commit 53c81c2); manuscript source in `../paper/`; check list in `../MANIFEST.md`; code map in `../MAP.md`.

**Source and code map**

| Result file | Producing script | What it holds |
|---|---|---|
| `results/loo_consistent_results.json` | `scripts/rerun_loo_consistent.py (BrainIAK, mpirun -np 1)` | SRM k, common-space and symmetric-LOSO disparity with Crawford-Howell tests (Figure 5, tab:disparity_loso) |
| `results/k_aggregation_results.json` | `scripts/k_selection/aggregate_k_selection.py` | mean-rank aggregation over folds and metrics (S4) |
| `results/crossnobis_results.json, pca_cca_results.json, ve_results.json` | `scripts/triangulation/*.py` | alignment-independent distances and variance explained (S9); files include sub-10, the notebook drops it |
| `results/overall_signal_results.json, activation_prior_results.json` | `scripts/activation/*.py` | activation-level metrics with single-case tests (S10) |
| `results/disparity_frozen_permutation_primary.json` | `scripts/color_specificity/disparity_frozen_permutation.py` | colour-correspondence permutation, frozen vs re-estimated projection (S18) |
| `results/color_correspondence_heldout.json` | `scripts/color_specificity/color_correspondence_heldout.py` | split-half RDM reliability (S18) |
| `results/cyclic_shift_disparity.json, shift_gain_ch.json` | `scripts/color_specificity/shift_gain_ch.py` | cyclic hue-shift disparity and gain test (S18) |

In [1]:
import sys, json, csv
from pathlib import Path
sys.path.insert(0, str((Path.cwd() / ".." / "common").resolve()))
import numpy as np
from scipy import stats
import verify as V
from stats_helpers import crawford_howell, hedges_g, bh_fdr, wilson_interval
R = Path("results")
def J(name):
    with open(R / name) as f:
        return json.load(f)
HC = [f"sub-{i:02d}" for i in range(1, 8)]
CVD = {"deutan": "sub-08", "protan": "sub-09"}
ROIS = ["V1", "V2", "V3", "hV4"]
HUES = ["red", "orange", "yellow", "green", "cyan", "blue", "purple", "magenta"]

loo = J("loo_consistent_results.json")
LOO = loo["results"]
SUBS = HC + ["sub-08", "sub-09"]
def disp_common(roi, sub):
    r = LOO[roi]; return r["hc_loo_disparities"][sub] if sub in HC else r["individual_cvd"][sub]["cvd_score"]

V.start("02_geometry")

### Procrustes disparity: both estimators (Results section 2; Figure 5; Supplementary tab:disparity_loso)
Crawford-Howell one-tailed t (df = 6) against the control leave-one-out distribution, in the common control-trained SRM space and under the symmetric LOSO estimator; d_cc = t·sqrt(8/7).

| id | manuscript | quantity | reported |
|---|---|---|---|
| 02.01 | tab:disparity_loso | 8 participant-by-ROI cells x (t, p, d_cc) x two estimators | `48 cells, see 02.T1.*` |
| 02.02 | Results §2 ¶1 | largest deviation is V1 in the protan participant (primary pipeline) | `V1` |
| 02.03 | Results §2 ¶1 | largest deviation is V2 in the deutan participant (primary pipeline) | `V2` |
| 02.04 | Results §2 ¶1 | under the symmetric LOSO reference only protan V1 reaches significance | `[('sub-09', 'V1')]` |
| 02.05 | Methods 'Shared Response Model' | seed, permutation counts recorded in the run config | `(42, 1000, 10000)` |

In [2]:
T = {("sub-08", "V1"): (1.10, 0.157, 1.18, 0.48, 0.323, 0.51), ("sub-08", "V2"): (2.11, 0.040, 2.26, 1.33, 0.116, 1.42),
     ("sub-08", "V3"): (1.92, 0.052, 2.05, 1.17, 0.143, 1.25), ("sub-08", "hV4"): (0.23, 0.411, 0.25, 0.07, 0.474, 0.07),
     ("sub-09", "V1"): (3.48, 0.007, 3.72, 2.02, 0.045, 2.16), ("sub-09", "V2"): (0.99, 0.181, 1.06, 0.77, 0.234, 0.82),
     ("sub-09", "V3"): (0.09, 0.466, 0.10, 0.06, 0.479, 0.06), ("sub-09", "hV4"): (1.13, 0.150, 1.21, 0.80, 0.228, 0.86)}
name = {"sub-08": "deutan", "sub-09": "protan"}
for (s, roi), (t_c, p_c, d_c, t_l, p_l, d_l) in T.items():
    c = LOO[roi]["individual_cvd"][s]; l = LOO[roi]["loso_analysis"]["individual_cvd"][s]
    V.check(f"02.T1.{name[s]}.{roi}.t", f"tab:disparity_loso {name[s]} {roi} common t", c["t_stat"], t_c, nd=2)
    V.check(f"02.T1.{name[s]}.{roi}.p", f"tab:disparity_loso {name[s]} {roi} common p", c["p_value"], p_c, nd=3)
    V.check(f"02.T1.{name[s]}.{roi}.d", f"tab:disparity_loso {name[s]} {roi} common d_cc", c["t_stat"] * np.sqrt(8 / 7), d_c, nd=2)
    V.check(f"02.T1.{name[s]}.{roi}.t_loso", f"tab:disparity_loso {name[s]} {roi} LOSO t", l["t_stat"], t_l, nd=2)
    V.check(f"02.T1.{name[s]}.{roi}.p_loso", f"tab:disparity_loso {name[s]} {roi} LOSO p", l["p_value"], p_l, nd=3)
    V.check(f"02.T1.{name[s]}.{roi}.d_loso", f"tab:disparity_loso {name[s]} {roi} LOSO d_cc", l["t_stat"] * np.sqrt(8 / 7), d_l, nd=2)
max_roi = {s: max(ROIS, key=lambda r: LOO[r]["individual_cvd"][s]["t_stat"]) for s in ("sub-08", "sub-09")}
loso_sig = [(s, r) for s in ("sub-08", "sub-09") for r in ROIS if LOO[r]["loso_analysis"]["individual_cvd"][s]["p_value"] < 0.05]
elevated = all(LOO[r]["individual_cvd"][s]["t_stat"] > 0 for s in ("sub-08", "sub-09") for r in ("V1", "V2", "V3", "hV4"))
print(max_roi, loso_sig)
V.table('02.01', 'tab:disparity_loso | 8 participant-by-ROI cells x (t, p, d_cc) x two estimators', '48 cells, see 02.T1.*')
V.check('02.02', 'Results §2 ¶1 | largest deviation is V1 in the protan participant (primary pipeline)', max_roi["sub-09"], 'V1', mode='eq')
V.check('02.03', 'Results §2 ¶1 | largest deviation is V2 in the deutan participant (primary pipeline)', max_roi["sub-08"], 'V2', mode='eq')
V.check('02.04', 'Results §2 ¶1 | under the symmetric LOSO reference only protan V1 reaches significance', loso_sig, [('sub-09', 'V1')], mode='eq')
V.check('02.05', "Methods 'Shared Response Model' | seed, permutation counts recorded in the run config", (loo["config"]["seed"], loo["config"]["n_perm_color"], loo["config"]["n_perm_group"]), (42, 1000, 10000), mode='eq')

[OK ] 02.T1.deutan.V1.t tab:disparity_loso deutan V1 common t: produced=1.101  reported=1.1
[OK ] 02.T1.deutan.V1.p tab:disparity_loso deutan V1 common p: produced=0.1566  reported=0.157
[OK ] 02.T1.deutan.V1.d tab:disparity_loso deutan V1 common d_cc: produced=1.177  reported=1.18
[OK ] 02.T1.deutan.V1.t_loso tab:disparity_loso deutan V1 LOSO t: produced=0.483  reported=0.48
[OK ] 02.T1.deutan.V1.p_loso tab:disparity_loso deutan V1 LOSO p: produced=0.3231  reported=0.323
[~~ ] 02.T1.deutan.V1.d_loso tab:disparity_loso deutan V1 LOSO d_cc: produced=0.5163  reported=0.51
[OK ] 02.T1.deutan.V2.t tab:disparity_loso deutan V2 common t: produced=2.113  reported=2.11
[OK ] 02.T1.deutan.V2.p tab:disparity_loso deutan V2 common p: produced=0.0395  reported=0.04
[OK ] 02.T1.deutan.V2.d tab:disparity_loso deutan V2 common d_cc: produced=2.259  reported=2.26
[OK ] 02.T1.deutan.V2.t_loso tab:disparity_loso deutan V2 LOSO t: produced=1.331  reported=1.33
[OK ] 02.T1.deutan.V2.p_loso tab:disparity_l

### Dimensionality of the shared response model (Supplementary S4)
k selected by mean rank over the seven LOSO folds and three metrics; the manuscript quotes the cross-participant RDM Spearman correlation at the selected k.

| id | manuscript | quantity | reported |
|---|---|---|---|
| 02.06 | S4 | k = 4 at V1 | `4` |
| 02.07 | S4 | k = 4 at V2 | `4` |
| 02.08 | S4 | k = 3 at V3 | `3` |
| 02.09 | S4 | k = 3 at hV4 | `3` |
| 02.10 | S4 | cross-participant RDM Spearman r at V1 | `0.6` |
| 02.11 | S4 | at V2 | `0.57` |
| 02.12 | S4 | at V3 | `0.55` |
| 02.13 | S4 | at hV4 | `0.32` |
| 02.14 | S4 | candidate range 2..6 over seven folds | `('2', '6', 7)` |

In [3]:
ka = J("k_aggregation_results.json")["results"]
kv_raw = loo["config"]["k_values"]
def _pick(d, r):
    for key in (r, r.replace("hV4", "V4"), r.replace("V4", "hV4"), r.lower()):
        if key in d:
            return d[key]
    raise KeyError(r)
kv = {r: (_pick(kv_raw, r) if isinstance(kv_raw, dict) else kv_raw[i]) for i, r in enumerate(ROIS)}
KA = {r: (r if r in ka else r.replace("hV4", "V4")) for r in ROIS}
xr = {r: ka[KA[r]]["cross_subject_rdm_corr"]["mean_values"][str(kv[r])] for r in ROIS}
print(kv, {r: round(v, 3) for r, v in xr.items()})
V.check('02.06', 'S4 | k = 4 at V1', kv["V1"], 4, mode='eq')
V.check('02.07', 'S4 | k = 4 at V2', kv["V2"], 4, mode='eq')
V.check('02.08', 'S4 | k = 3 at V3', kv["V3"], 3, mode='eq')
V.check('02.09', 'S4 | k = 3 at hV4', kv["hV4"], 3, mode='eq')
V.check('02.10', 'S4 | cross-participant RDM Spearman r at V1', xr["V1"], 0.6, nd=2)
V.check('02.11', 'S4 | at V2', xr["V2"], 0.57, nd=2)
V.check('02.12', 'S4 | at V3', xr["V3"], 0.55, nd=2)
V.check('02.13', 'S4 | at hV4', xr["hV4"], 0.32, nd=2)
V.check('02.14', 'S4 | candidate range 2..6 over seven folds', (min(ka["V1"]["cross_subject_rdm_corr"]["mean_values"].keys()), max(ka["V1"]["cross_subject_rdm_corr"]["mean_values"].keys()), ka["V1"]["n_folds"]), ('2', '6', 7), mode='eq')

{'V1': 4, 'V2': 4, 'V3': 3, 'hV4': 3} {'V1': 0.596, 'V2': 0.566, 'V3': 0.546, 'hV4': 0.317}
[OK ] 02.06 S4 | k = 4 at V1: produced=4  reported=4
[OK ] 02.07 S4 | k = 4 at V2: produced=4  reported=4
[OK ] 02.08 S4 | k = 3 at V3: produced=3  reported=3
[OK ] 02.09 S4 | k = 3 at hV4: produced=3  reported=3
[OK ] 02.10 S4 | cross-participant RDM Spearman r at V1: produced=0.5965  reported=0.6
[OK ] 02.11 S4 | at V2: produced=0.5656  reported=0.57
[OK ] 02.12 S4 | at V3: produced=0.5464  reported=0.55
[OK ] 02.13 S4 | at hV4: produced=0.3165  reported=0.32
[OK ] 02.14 S4 | candidate range 2..6 over seven folds: produced=(2, 6, 7)  reported=(2, 6, 7)


### Alignment-independent checks (Supplementary S9, tab:triangulation)
Spearman r over the nine analysed participants between SRM disparity and (i) crossnobis distance in native voxel space, (ii) PCA and (iii) PCA-CCA pairwise distance; pooled = all four ROIs concatenated. The committed files include sub-10, which is dropped here.

| id | manuscript | quantity | reported |
|---|---|---|---|
| 02.15 | tab:triangulation | Spearman r, 3 distances x (4 ROIs + pooled) | `15 cells, see 02.T2.*` |
| 02.16 | S9 ¶1 | crossnobis pooled p < .001 | `0.001` |
| 02.17 | S9 ¶1 | PCA pooled p < .001 | `0.001` |
| 02.18 | S9 ¶3 | PCA-CCA pooled p < .001 | `0.001` |
| 02.19 | S9 ¶2 | crossnobis V1 convergent r p = .005 | `0.005` |
| 02.20 | S9 ¶2 | crossnobis V2 convergent r p = .025 | `0.025` |
| 02.21 | S9 ¶2 | control-to-control minus control-to-CVD crossnobis RDM similarity at V1 | `0.104` |
| 02.22 | S9 ¶2 | the three remaining ROIs give differences below 0.05 | `0.05` |
| 02.23 | S9 ¶2 | permutation p = .120 for the V1 difference | `the committed permutation was run with sub-10 included (p = .051); the n = 2 permutation needs the per-participant crossnobis RDMs, which are not stored` |
| 02.24 | S9 ¶3 | Hedges g under PCA ranges from -0.13 to +0.40 | `(-0.13, 0.4)` |
| 02.25 | S9 ¶3 | Hedges g under PCA-CCA ranges from -0.13 to +0.16 | `(-0.13, 0.16)` |

In [4]:
cn = J("crossnobis_results.json")["results"]; pc = J("pca_cca_results.json")["results"]
pooled = {k: ([], []) for k in ("cn", "pca", "cca")}; tri = {}
for roi in ROIS:
    a = [disp_common(roi, s) for s in SUBS]
    b = [cn[roi]["distance_from_hc_mean"][s] for s in SUBS]
    c = [pc[roi]["pca_only"]["subject_mean_dist_to_hc"][s] for s in SUBS]
    d = [pc[roi]["pca_cca"]["subject_mean_dist_to_hc"][s] for s in SUBS]
    tri[roi] = {"cn": stats.spearmanr(a, b), "pca": stats.spearmanr(a, c), "cca": stats.spearmanr(a, d)}
    for k, v in zip(("cn", "pca", "cca"), (b, c, d)):
        pooled[k][0].extend(a); pooled[k][1].extend(v)
pool = {k: stats.spearmanr(*pooled[k]) for k in pooled}
T = {"cn": (0.833, 0.733, 0.550, 0.300, 0.632), "pca": (0.633, 0.850, 0.333, 0.533, 0.780), "cca": (0.483, 0.433, 0.100, -0.067, 0.544)}
LAB = {"cn": "Crossnobis", "pca": "PCA", "cca": "PCA-CCA"}
for k, vals in T.items():
    for roi, rep in zip(ROIS, vals[:4]):
        V.check(f"02.T2.{k}.{roi}", f"tab:triangulation {LAB[k]} {roi}", tri[roi][k][0], rep, nd=3)
    V.check(f"02.T2.{k}.pooled", f"tab:triangulation {LAB[k]} pooled", pool[k][0], vals[4], nd=3)
# crossnobis RDM-similarity difference recomputed without sub-10 (HC-major order, sub-10 third in each block)
cn_diff = {}
for roi in ROIS:
    v = cn[roi]["rdm_similarities"]; sub = [x for i, x in enumerate(v["hc_cvd"]["values"]) if i % 3 != 2]
    cn_diff[roi] = np.mean(v["hc_hc"]["values"]) - np.mean(sub)
# Hedges g (CVD minus control similarity) without sub-10
def g_n2(block):
    sub = [x for i, x in enumerate(block["hc_cvd"]["values"]) if i % 3 != 2]
    return hedges_g(sub, block["hc_hc"]["values"])
g_pca = [g_n2(pc[r]["pca_only"]) for r in ROIS]; g_cca = [g_n2(pc[r]["pca_cca"]) for r in ROIS]
print({r: round(v, 3) for r, v in cn_diff.items()}, np.round(g_pca, 2), np.round(g_cca, 2))
V.table('02.15', 'tab:triangulation | Spearman r, 3 distances x (4 ROIs + pooled)', '15 cells, see 02.T2.*')
V.check('02.16', 'S9 ¶1 | crossnobis pooled p < .001', pool["cn"][1], 0.001, mode='lt')
V.check('02.17', 'S9 ¶1 | PCA pooled p < .001', pool["pca"][1], 0.001, mode='lt')
V.check('02.18', 'S9 ¶3 | PCA-CCA pooled p < .001', pool["cca"][1], 0.001, mode='lt')
V.check('02.19', 'S9 ¶2 | crossnobis V1 convergent r p = .005', tri["V1"]["cn"][1], 0.005, nd=3)
V.check('02.20', 'S9 ¶2 | crossnobis V2 convergent r p = .025', tri["V2"]["cn"][1], 0.025, nd=3)
V.check('02.21', 'S9 ¶2 | control-to-control minus control-to-CVD crossnobis RDM similarity at V1', cn_diff["V1"], 0.104, nd=3)
V.check('02.22', 'S9 ¶2 | the three remaining ROIs give differences below 0.05', max(cn_diff[r] for r in ("V2", "V3", "hV4")), 0.05, mode='lt')
V.flag('02.23', 'S9 ¶2 | permutation p = .120 for the V1 difference', 'the committed permutation was run with sub-10 included (p = .051); the n = 2 permutation needs the per-participant crossnobis RDMs, which are not stored', 'the committed permutation was run with sub-10 included (p = .051); the n = 2 permutation needs the per-participant crossnobis RDMs, which are not stored')
V.check('02.24', 'S9 ¶3 | Hedges g under PCA ranges from -0.13 to +0.40', (min(g_pca), max(g_pca)), (-0.13, 0.4), mode='pair', nd=2)
V.check('02.25', 'S9 ¶3 | Hedges g under PCA-CCA ranges from -0.13 to +0.16', (min(g_cca), max(g_cca)), (-0.13, 0.16), mode='pair', nd=2)

[OK ] 02.T2.cn.V1 tab:triangulation Crossnobis V1: produced=0.8333  reported=0.833
[OK ] 02.T2.cn.V2 tab:triangulation Crossnobis V2: produced=0.7333  reported=0.733
[OK ] 02.T2.cn.V3 tab:triangulation Crossnobis V3: produced=0.55  reported=0.55
[OK ] 02.T2.cn.hV4 tab:triangulation Crossnobis hV4: produced=0.3  reported=0.3
[OK ] 02.T2.cn.pooled tab:triangulation Crossnobis pooled: produced=0.6324  reported=0.632
[OK ] 02.T2.pca.V1 tab:triangulation PCA V1: produced=0.6333  reported=0.633
[OK ] 02.T2.pca.V2 tab:triangulation PCA V2: produced=0.85  reported=0.85
[OK ] 02.T2.pca.V3 tab:triangulation PCA V3: produced=0.3333  reported=0.333
[OK ] 02.T2.pca.hV4 tab:triangulation PCA hV4: produced=0.5333  reported=0.533
[OK ] 02.T2.pca.pooled tab:triangulation PCA pooled: produced=0.7804  reported=0.78
[OK ] 02.T2.cca.V1 tab:triangulation PCA-CCA V1: produced=0.4833  reported=0.483
[OK ] 02.T2.cca.V2 tab:triangulation PCA-CCA V2: produced=0.4333  reported=0.433
[OK ] 02.T2.cca.V3 tab:triangu

### Variance explained under the symmetric LOSO projection (Supplementary S9, tab:variance_explained)
Hedges g is signed control minus CVD (n = 7 vs n = 2).

| id | manuscript | quantity | reported |
|---|---|---|---|
| 02.26 | tab:variance_explained | 4 ROIs x (k, controls, CVD, g) | `16 cells, see 02.T3.*` |
| 02.27 | S9 ¶4 | variance explained higher in CVD than controls at all four ROIs | `True` |
| 02.28 | S9 ¶4 | variance explained vs disparity, pooled Spearman r = -0.214 (common-space disparity, 36 points) | `-0.214` |
| 02.29 | S9 ¶4 | p = .211 | `0.211` |

In [5]:
ve = J("ve_results.json")["results"]
VK = {"V1": "V1", "V2": "V2", "V3": "V3", "hV4": "hV4" if "hV4" in ve else "V4"}
T = {"V1": (4, 0.352, 0.407, -0.40), "V2": (4, 0.331, 0.416, -1.26), "V3": (3, 0.250, 0.314, -0.66), "hV4": (3, 0.225, 0.253, -0.39)}
ve_hc = {}; ve_cvd = {}
for roi, (k_r, hc_r, cvd_r, g_r) in T.items():
    b = ve[VK[roi]]["framework_b"]
    hc = [b["hc_ve"][s] for s in HC]; cv = [b["cvd_ve"][s] for s in ("sub-08", "sub-09")]
    ve_hc[roi] = np.mean(hc); ve_cvd[roi] = np.mean(cv)
    V.check(f"02.T3.{roi}.k", f"tab:variance_explained {roi} k", ve[VK[roi]]["k"], k_r, mode="eq")
    V.check(f"02.T3.{roi}.controls", f"tab:variance_explained {roi} controls", np.mean(hc), hc_r, nd=3)
    V.check(f"02.T3.{roi}.cvd", f"tab:variance_explained {roi} CVD", np.mean(cv), cvd_r, nd=3)
    V.check(f"02.T3.{roi}.g", f"tab:variance_explained {roi} Hedges g", hedges_g(hc, cv), g_r, nd=2)
ve_all, dsp_all = [], []
for roi in ROIS:
    b = ve[VK[roi]]["framework_b"]
    for s in SUBS:
        ve_all.append(b["hc_ve"][s] if s in HC else b["cvd_ve"][s]); dsp_all.append(disp_common(roi, s))
r_ve, p_ve = stats.spearmanr(ve_all, dsp_all)   # the manuscript's pooled correlation is a Spearman r over 36 participant-by-ROI points
print(r_ve, p_ve)
V.table('02.26', 'tab:variance_explained | 4 ROIs x (k, controls, CVD, g)', '16 cells, see 02.T3.*')
V.check('02.27', 'S9 ¶4 | variance explained higher in CVD than controls at all four ROIs', all(ve_cvd[r] > ve_hc[r] for r in ROIS), True, mode='eq')
V.check('02.28', 'S9 ¶4 | variance explained vs disparity, pooled Spearman r = -0.214 (common-space disparity, 36 points)', r_ve, -0.214, nd=3)
V.check('02.29', 'S9 ¶4 | p = .211', p_ve, 0.211, nd=3)

[OK ] 02.T3.V1.k tab:variance_explained V1 k: produced=4  reported=4
[OK ] 02.T3.V1.controls tab:variance_explained V1 controls: produced=0.3524  reported=0.352
[OK ] 02.T3.V1.cvd tab:variance_explained V1 CVD: produced=0.4074  reported=0.407
[OK ] 02.T3.V1.g tab:variance_explained V1 Hedges g: produced=-0.3976  reported=-0.4
[OK ] 02.T3.V2.k tab:variance_explained V2 k: produced=4  reported=4
[OK ] 02.T3.V2.controls tab:variance_explained V2 controls: produced=0.3312  reported=0.331
[OK ] 02.T3.V2.cvd tab:variance_explained V2 CVD: produced=0.4163  reported=0.416
[OK ] 02.T3.V2.g tab:variance_explained V2 Hedges g: produced=-1.257  reported=-1.26
[OK ] 02.T3.V3.k tab:variance_explained V3 k: produced=3  reported=3
[OK ] 02.T3.V3.controls tab:variance_explained V3 controls: produced=0.2503  reported=0.25
[OK ] 02.T3.V3.cvd tab:variance_explained V3 CVD: produced=0.3138  reported=0.314
[OK ] 02.T3.V3.g tab:variance_explained V3 Hedges g: produced=-0.6574  reported=-0.66
[OK ] 02.T3.hV4.

### Activation-level comparison (Supplementary S10)
Two-tailed single-case tests on the pre-SRM amplitudes; the metric-disparity correlations come from the activation-prior analysis.

| id | manuscript | quantity | reported |
|---|---|---|---|
| 02.30 | S10 ¶2 | mean |beta|: every single-case p >= 0.182 | `0.182` |
| 02.31 | S10 ¶2 | largest mean |beta| deviation is the deutan participant at V3 | `('sub-08', 'V3')` |
| 02.32 | S10 ¶2 | its d_cc = +1.61 | `1.61` |
| 02.33 | S10 ¶2 | largest modulation-depth deviation is the deutan participant at V2 | `('sub-08', 'V2')` |
| 02.34 | S10 ¶2 | its d_cc = +2.16 | `2.16` |
| 02.35 | S10 ¶2 | its p = 0.090 | `0.09` |
| 02.36 | S10 ¶2 | voxel-level colour selectivity F > 4 in V1-V3 (mean F over the nine participants; one participant's V3 F is 1.46, p = .18, so the statement holds at the group level) | `4.0` |
| 02.37 | S10 ¶2 | p < 0.001 for every participant in V1 and V2 | `0.001` |
| 02.38 | S10 ¶3 | activation metrics vs disparity: 20 tests, r from -0.29 to +0.51, all p >= 0.130 | `(20, -0.29, 0.51, 0.13)` |

In [6]:
os_ = J("overall_signal_results.json")
mabs = {(s, r): os_[r]["individual_tests"][s]["mean_abs_act"] for s in ("sub-08", "sub-09") for r in ROIS}
mod = {(s, r): os_[r]["individual_tests"][s]["modulation_depth"] for s in ("sub-08", "sub-09") for r in ROIS}
min_p_mabs = min(v["p"] for v in mabs.values())
top_mabs = max(mabs.items(), key=lambda kv: kv[1]["zcc"]); top_mod = max(mod.items(), key=lambda kv: kv[1]["zcc"])
F_mean = {r: np.mean([os_[r]["per_subject"][s]["color_selectivity_F"] for s in SUBS]) for r in ROIS}
F_min = {r: min(os_[r]["per_subject"][s]["color_selectivity_F"] for s in SUBS) for r in ROIS}
Fp_max = {r: max(os_[r]["per_subject"][s]["color_selectivity_p"] for s in SUBS) for r in ROIS}
print("colour-selectivity F: mean over participants", {r: round(v, 1) for r, v in F_mean.items()}, "; participant minimum", {r: round(v, 2) for r, v in F_min.items()}, "; largest participant p", {r: round(v, 3) for r, v in Fp_max.items()})
ap = J("activation_prior_results.json")
corr_r, corr_p = [], []
for roi, blk in ap.items():
    for k, v in blk.items():
        if isinstance(v, dict) and "correlation_with_disparity" in v:
            corr_r.append(v["correlation_with_disparity"]["r"]); corr_p.append(v["correlation_with_disparity"]["p"])
if not corr_r:   # fall back to the stored top-level layout, if different
    def walk(o):
        if isinstance(o, dict):
            if "r" in o and "p" in o and len(o) <= 4:
                yield o["r"], o["p"]
            for v in o.values():
                yield from walk(v)
    for roi, blk in ap.items():
        for key in blk:
            if "corr" in key.lower():
                for r_, p_ in walk(blk[key]):
                    corr_r.append(r_); corr_p.append(p_)
print(top_mabs, top_mod, F_min, Fp_max, len(corr_r))
V.check('02.30', 'S10 ¶2 | mean |beta|: every single-case p >= 0.182', min_p_mabs, 0.182, nd=3)
V.check('02.31', 'S10 ¶2 | largest mean |beta| deviation is the deutan participant at V3', top_mabs[0], ('sub-08', 'V3'), mode='eq')
V.check('02.32', 'S10 ¶2 | its d_cc = +1.61', top_mabs[1]["zcc"], 1.61, nd=2)
V.check('02.33', 'S10 ¶2 | largest modulation-depth deviation is the deutan participant at V2', top_mod[0], ('sub-08', 'V2'), mode='eq')
V.check('02.34', 'S10 ¶2 | its d_cc = +2.16', top_mod[1]["zcc"], 2.16, nd=2)
V.check('02.35', 'S10 ¶2 | its p = 0.090', top_mod[1]["p"], 0.09, nd=3)
V.check('02.36', "S10 ¶2 | voxel-level colour selectivity F > 4 in V1-V3 (mean F over the nine participants; one participant's V3 F is 1.46, p = .18, so the statement holds at the group level)", min(F_mean[r] for r in ("V1", "V2", "V3")), 4.0, mode='gt')
V.check('02.37', 'S10 ¶2 | p < 0.001 for every participant in V1 and V2', max(Fp_max["V1"], Fp_max["V2"]), 0.001, mode='lt')
V.check('02.38', 'S10 ¶3 | activation metrics vs disparity: 20 tests, r from -0.29 to +0.51, all p >= 0.130', (len(corr_r), round(min(corr_r), 2), round(max(corr_r), 2), round(min(corr_p), 3)) if corr_r else None, (20, -0.29, 0.51, 0.13), mode='eq_or_flag', reason='activation_prior_results.json does not store the metric-disparity correlations in a recognised layout')

colour-selectivity F: mean over participants {'V1': 37.0, 'V2': 25.7, 'V3': 10.2, 'hV4': 10.6} ; participant minimum {'V1': 9.92, 'V2': 5.38, 'V3': 1.46, 'hV4': 2.0} ; largest participant p {'V1': 0.0, 'V2': 0.0, 'V3': 0.176, 'hV4': 0.053}
(('sub-08', 'V3'), {'value': 0.017523528879981508, 'zcc': 1.613179661269465, 'p': 0.1820357759101655}) (('sub-08', 'V2'), {'value': 0.010988334478323914, 'zcc': 2.1626194047581153, 'p': 0.08953439880306585}) {'V1': 9.918701847496642, 'V2': 5.376835553528279, 'V3': 1.4644761499999301, 'hV4': 2.00205172640157} {'V1': 2.8382698742267203e-12, 'V2': 3.857660722492254e-06, 'V3': 0.1763358824861971, 'hV4': 0.053046559770684044} 0
[OK ] 02.30 S10 ¶2 | mean |beta|: every single-case p >= 0.182: produced=0.182  reported=0.182
[OK ] 02.31 S10 ¶2 | largest mean |beta| deviation is the deutan participant at V3: produced=(sub-08, V3)  reported=(sub-08, V3)
[OK ] 02.32 S10 ¶2 | its d_cc = +1.61: produced=1.613  reported=1.61
[OK ] 02.33 S10 ¶2 | largest modulation-

### Colour specificity of the geometry (Supplementary S18)
Colour-correspondence permutation on the controls as a positive control (tab:frozen_control) and the participant-by-region grid under the frozen projection (tab:color_specificity, primary pipeline), with Benjamini-Hochberg correction over the 35 cells.

| id | manuscript | quantity | reported |
|---|---|---|---|
| 02.39 | tab:frozen_control | 4 ROIs x (n, detected and mean z under re-estimated and frozen projections) | `20 cells, see 02.T4.*` |
| 02.40 | tab:color_specificity | 35 participant-by-region p_perm cells, primary pipeline | `35 cells, see 02.T5.*` |
| 02.41 | S18 ¶2 | cells below .05 out of 35 | `16` |
| 02.42 | S18 ¶2 | chance expectation 35 x 0.05 | `1.8` |
| 02.43 | S18 ¶2 | cells surviving BH over 35 | `7` |
| 02.44 | S18 ¶2 | deutan V2 q | `0.018` |
| 02.45 | S18 ¶2 | protan V3 q | `0.012` |
| 02.46 | S18 ¶2 | the two CVD survivors are deutan V2 and protan V3 | `[('sub-08', 'V2'), ('sub-09', 'V3')]` |
| 02.47 | S18 ¶2 | V3: 5 of its 9 cells survive | `5` |
| 02.48 | S18 ¶2 | deutan V2 is the lowest V2 value among the nine participants | `sub-08` |
| 02.49 | S18 ¶1 | number of permutations | `1000` |

In [7]:
fz = J("disparity_frozen_permutation_primary.json")["results"]
T_FC = {"V1": (7, 0, -0.04, 2, -1.58), "V2": (7, 0, -0.04, 3, -1.00), "V3": (7, 0, -0.78, 5, -2.39), "hV4": (6, 0, 0.06, 2, -1.07)}
for roi, (n_r, det_re, z_re, det_fr, z_fr) in T_FC.items():
    re = fz[roi]["modes"]["refit_projection"]["hc"]; fr = fz[roi]["modes"]["frozen_projection"]["hc"]
    V.check(f"02.T4.{roi}.n", f"tab:frozen_control {roi} n", fz[roi]["n_hc"], n_r, mode="eq")
    V.check(f"02.T4.{roi}.refit_detected", f"tab:frozen_control {roi} re-estimated detected", re["n_p_perm_lt_05"], det_re, mode="eq")
    V.check(f"02.T4.{roi}.refit_z", f"tab:frozen_control {roi} re-estimated mean z", np.mean([v["z"] for v in re["per_subject"].values()]), z_re, nd=2)
    V.check(f"02.T4.{roi}.frozen_detected", f"tab:frozen_control {roi} frozen detected", fr["n_p_perm_lt_05"], det_fr, mode="eq")
    V.check(f"02.T4.{roi}.frozen_z", f"tab:frozen_control {roi} frozen mean z", np.mean([v["z"] for v in fr["per_subject"].values()]), z_fr, nd=2)
T_CS = {"Control 1": (0.088, 0.020, 0.062, 0.393), "Control 2": (0.001, 0.016, 0.004, 0.136), "Control 3": (0.109, 0.520, 0.255, 0.272),
        "Control 4": (0.152, 0.014, 0.001, 0.041), "Control 5": (0.057, 0.773, 0.034, 0.031), "Control 6": (0.407, 0.290, 0.009, 0.220),
        "Control 7": (0.034, 0.159, 0.004, None), "Deutan": (0.105, 0.002, 0.024, 0.273), "Protan": (0.758, 0.013, 0.001, 0.129)}
SUBMAP = {f"Control {i}": f"sub-{i:02d}" for i in range(1, 8)}; SUBMAP.update({"Deutan": "sub-08", "Protan": "sub-09"})
cells = {}
for lab, vals in T_CS.items():
    s = SUBMAP[lab]
    for roi, rep in zip(ROIS, vals):
        fr = fz[roi]["modes"]["frozen_projection"]
        rec = fr["hc"]["per_subject"].get(s) or fr["cvd"].get(s)
        if rep is None:
            V.check(f"02.T5.{lab}.{roi}", f"tab:color_specificity {lab} {roi} (absent: too few voxels)", rec is None, True, mode="eq"); continue
        cells[(s, roi)] = rec["p_perm"]
        V.check(f"02.T5.{lab}.{roi}", f"tab:color_specificity {lab} {roi} p_perm", rec["p_perm"], rep, nd=3)
labels = list(cells.keys()); p_arr = np.array([cells[k] for k in labels]); q_arr = bh_fdr(p_arr)
q = dict(zip(labels, q_arr))
n_raw = int((p_arr < 0.05).sum()); n_bh = int((q_arr < 0.05).sum())
v3_surv = sum(q[(s, "V3")] < 0.05 for s in SUBS)
cvd_surv = [k for k in labels if k[0] in ("sub-08", "sub-09") and q[k] < 0.05]
print(n_raw, n_bh, v3_surv, cvd_surv)
V.table('02.39', 'tab:frozen_control | 4 ROIs x (n, detected and mean z under re-estimated and frozen projections)', '20 cells, see 02.T4.*')
V.table('02.40', 'tab:color_specificity | 35 participant-by-region p_perm cells, primary pipeline', '35 cells, see 02.T5.*')
V.check('02.41', 'S18 ¶2 | cells below .05 out of 35', n_raw, 16, mode='eq')
V.check('02.42', 'S18 ¶2 | chance expectation 35 x 0.05', 35 * 0.05, 1.8, nd=1)
V.check('02.43', 'S18 ¶2 | cells surviving BH over 35', n_bh, 7, mode='eq')
V.check('02.44', 'S18 ¶2 | deutan V2 q', q[("sub-08", "V2")], 0.018, nd=3)
V.check('02.45', 'S18 ¶2 | protan V3 q', q[("sub-09", "V3")], 0.012, nd=3)
V.check('02.46', 'S18 ¶2 | the two CVD survivors are deutan V2 and protan V3', sorted(cvd_surv), [('sub-08', 'V2'), ('sub-09', 'V3')], mode='eq')
V.check('02.47', 'S18 ¶2 | V3: 5 of its 9 cells survive', v3_surv, 5, mode='eq')
V.check('02.48', 'S18 ¶2 | deutan V2 is the lowest V2 value among the nine participants', min(SUBS, key=lambda s: cells[(s, "V2")]), 'sub-08', mode='eq')
V.check('02.49', 'S18 ¶1 | number of permutations', J("disparity_frozen_permutation_primary.json")["meta"]["n_perm"], 1000, mode='eq')

[OK ] 02.T4.V1.n tab:frozen_control V1 n: produced=7  reported=7
[OK ] 02.T4.V1.refit_detected tab:frozen_control V1 re-estimated detected: produced=0  reported=0
[OK ] 02.T4.V1.refit_z tab:frozen_control V1 re-estimated mean z: produced=-0.04043  reported=-0.04
[OK ] 02.T4.V1.frozen_detected tab:frozen_control V1 frozen detected: produced=2  reported=2
[OK ] 02.T4.V1.frozen_z tab:frozen_control V1 frozen mean z: produced=-1.58  reported=-1.58
[OK ] 02.T4.V2.n tab:frozen_control V2 n: produced=7  reported=7
[OK ] 02.T4.V2.refit_detected tab:frozen_control V2 re-estimated detected: produced=0  reported=0
[OK ] 02.T4.V2.refit_z tab:frozen_control V2 re-estimated mean z: produced=-0.04286  reported=-0.04
[OK ] 02.T4.V2.frozen_detected tab:frozen_control V2 frozen detected: produced=3  reported=3
[OK ] 02.T4.V2.frozen_z tab:frozen_control V2 frozen mean z: produced=-1.002  reported=-1
[OK ] 02.T4.V3.n tab:frozen_control V3 n: produced=7  reported=7
[OK ] 02.T4.V3.refit_detected tab:frozen_

### The protan V1 correspondence is displaced by one hue step (Supplementary S18, last paragraphs)
Split-half reliability, cyclic hue-shift disparity and the shift-gain test against the control gain distribution (same minimum-over-eight rule).

| id | manuscript | quantity | reported |
|---|---|---|---|
| 02.50 | S18 ¶4 | protan V1 split-half pattern reliability | `0.847` |
| 02.51 | S18 ¶4 | above the highest control value | `True` |
| 02.52 | S18 ¶4 | eight-way classification 0.79 at protan V1 | `readout not stored in a committed artifact (the frozen-permutation driver printed it to stdout)` |
| 02.53 | S18 ¶4 | protan V1 disparity with identity labels | `1.037` |
| 02.54 | S18 ¶4 | after a 45-degree rotation of the labels | `0.788` |
| 02.55 | S18 ¶4 | gain (%) | `24.0` |
| 02.56 | S18 ¶4 | control gain mean (%) | `3.5` |
| 02.57 | S18 ¶4 | control gain SD (%) | `5.9` |
| 02.58 | S18 ¶4 | Crawford-Howell t of the protan gain | `3.22` |
| 02.59 | S18 ¶4 | p | `0.009` |
| 02.60 | S18 ¶4 | d_cc | `3.44` |
| 02.61 | S18 ¶4 | control mean disparity at V1 | `0.839` |
| 02.62 | S18 ¶4 | control SD | `0.087` |
| 02.63 | S18 ¶4 | rotated protan geometry lies below the control mean | `True` |
| 02.64 | S18 ¶4 | second-order RSA rho 0.00 -> +0.52 (controls 0.45), z = +5.02, p = .002, p_adj = .032; SRM 225-degree shift +0.50; PCA optimum at 135 degrees p = .048 | `no committed artifact for the second-order RSA cyclic-shift statistics` |
| 02.65 | S18 ¶4 | deutan: identity mapping optimal at V1, V2 and V3 | `True` |

In [8]:
cch = J("color_correspondence_heldout.json")["results"]["V1"]["within_subject_split_half_rdm_r"]
cs = J("cyclic_shift_disparity.json")["V1"]; sg = J("shift_gain_ch.json")["arms"]["with_residuals"]
hc_max_split = max(cch[s] for s in HC)
d0 = cs["sub-09"][0]; d45 = cs["sub-09"][1]; gain = 1 - d45 / d0
hc_d0 = np.array([cs[s][0] for s in HC])
protan_v1 = sg["V1"]["cvd"]["protan"]
deutan_identity = all(sg[r]["cvd"]["deutan"]["best_shift_idx"] == 0 for r in ("V1", "V2", "V3"))
print(cch["sub-09"], hc_max_split, d0, d45, gain, protan_v1, hc_d0.mean(), hc_d0.std(ddof=1))
V.check('02.50', 'S18 ¶4 | protan V1 split-half pattern reliability', cch["sub-09"], 0.847, nd=3)
V.check('02.51', 'S18 ¶4 | above the highest control value', cch["sub-09"] > hc_max_split, True, mode='eq')
V.flag('02.52', 'S18 ¶4 | eight-way classification 0.79 at protan V1', 'readout not stored in a committed artifact (the frozen-permutation driver printed it to stdout)', 'readout not stored in a committed artifact (the frozen-permutation driver printed it to stdout)')
V.check('02.53', 'S18 ¶4 | protan V1 disparity with identity labels', d0, 1.037, nd=3)
V.check('02.54', 'S18 ¶4 | after a 45-degree rotation of the labels', d45, 0.788, nd=3)
V.check('02.55', 'S18 ¶4 | gain (%)', gain * 100, 24.0, nd=1)
V.check('02.56', 'S18 ¶4 | control gain mean (%)', sg["V1"]["hc_gain_mean"] * 100, 3.5, nd=1)
V.check('02.57', 'S18 ¶4 | control gain SD (%)', sg["V1"]["hc_gain_sd"] * 100, 5.9, nd=1)
V.check('02.58', 'S18 ¶4 | Crawford-Howell t of the protan gain', protan_v1["t"], 3.22, nd=2)
V.check('02.59', 'S18 ¶4 | p', protan_v1["p_one_tailed_upper"], 0.009, nd=3)
V.check('02.60', 'S18 ¶4 | d_cc', protan_v1["d_cc"], 3.44, nd=2)
V.check('02.61', 'S18 ¶4 | control mean disparity at V1', hc_d0.mean(), 0.839, nd=3)
V.check('02.62', 'S18 ¶4 | control SD', hc_d0.std(ddof=1), 0.087, nd=3)
V.check('02.63', 'S18 ¶4 | rotated protan geometry lies below the control mean', d45 < hc_d0.mean(), True, mode='eq')
V.flag('02.64', 'S18 ¶4 | second-order RSA rho 0.00 -> +0.52 (controls 0.45), z = +5.02, p = .002, p_adj = .032; SRM 225-degree shift +0.50; PCA optimum at 135 degrees p = .048', 'no committed artifact for the second-order RSA cyclic-shift statistics', 'no committed artifact for the second-order RSA cyclic-shift statistics')
V.check('02.65', 'S18 ¶4 | deutan: identity mapping optimal at V1, V2 and V3', deutan_identity, True, mode='eq')

0.847 0.8 1.0369 0.7884 0.23965666891696402 {'gain': 0.239656668916964, 'best_shift_idx': 1, 'best_shift_deg': 45, 't': 3.2194250338118633, 'p_one_tailed_upper': 0.00907515837370243, 'd_cc': 3.4417101311220297} 0.8388999999999999 0.08673511399658156
[OK ] 02.50 S18 ¶4 | protan V1 split-half pattern reliability: produced=0.847  reported=0.847
[OK ] 02.51 S18 ¶4 | above the highest control value: produced=True  reported=True
[-- ] 02.52 S18 ¶4 | eight-way classification 0.79 at protan V1: reported=readout not stored in a committed artifact (the frozen-permutation driver printed it to stdout)  NO COMMITTED ARTIFACT: readout not stored in a committed artifact (the frozen-permutation driver printed it to stdout)
[OK ] 02.53 S18 ¶4 | protan V1 disparity with identity labels: produced=1.037  reported=1.037
[OK ] 02.54 S18 ¶4 | after a 45-degree rotation of the labels: produced=0.7884  reported=0.788
[OK ] 02.55 S18 ¶4 | gain (%): produced=23.97  reported=24
[OK ] 02.56 S18 ¶4 | control gain m

In [9]:
V.summary()


=== 02_geometry: 187/191 numeric checks reproduced exactly; 4 within one unit of the last printed digit; 0 mismatch, 0 error, 4 pointer-only ===
  NEAR     02.T1.deutan.V1.d_loso tab:disparity_loso deutan V1 LOSO d_cc: produced=0.5163 reported=0.51
  NEAR     02.T1.protan.V2.d tab:disparity_loso protan V2 common d_cc: produced=1.054 reported=1.06
  NEAR     02.T1.protan.V2.d_loso tab:disparity_loso protan V2 LOSO d_cc: produced=0.8263 reported=0.82
  NEAR     02.T1.protan.hV4.d_loso tab:disparity_loso protan hV4 LOSO d_cc: produced=0.8517 reported=0.86
